## 2 - Clean and Transform
ვკითხულობთ getdata.raw.ecommerce_orders-ს და ვქმნით თეიბლებს getdata.calculated-ში:

- შეკვეთები (orders): ტრანზაქციის დონის ფაქტები, გასუფთავებული ტიპები, წაშლილი ზედმეტი/დუბლირებული სვეტები
- მომხმარებლები (customers): დუბლიკატებისგან გასუფთავებული მომხმარებლის განზომილება (განზომილების ცხრილი)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS getdata.calculated")

DataFrame[]

In [0]:
import pyspark.sql.functions as F

raw_df = spark.table("getdata.raw.ecommerce_orders")

In [0]:
orders_df = (
    raw_df
    # კონვერტირება სტანდარტიზირებული კი/არა სვეტში
    .withColumn("Returned",       F.col("Returned")       == "Yes")
    .withColumn("Coupon_Used",    F.col("Coupon_Used")    == "Yes")
    .withColumn("Holiday_Season", F.col("Holiday_Season") == "Yes")
    .withColumn("High_Value_Order", F.col("High_Value_Order") == "Yes")

    # ზედმეტი ჰარების მოშორება
    .withColumn("Customer_ID",       F.trim(F.col("Customer_ID")))
    .withColumn("Customer_Gender",   F.trim(F.col("Customer_Gender")))
    .withColumn("Country",           F.trim(F.col("Country")))
    .withColumn("City",              F.trim(F.col("City")))
    .withColumn("Customer_Segment",  F.trim(F.col("Customer_Segment")))
    .withColumn("Product_ID",        F.trim(F.col("Product_ID")))
    .withColumn("Product_Category",  F.trim(F.col("Product_Category")))
    .withColumn("Product_Subcategory", F.trim(F.col("Product_Subcategory")))
    .withColumn("Brand",             F.trim(F.col("Brand")))
    .withColumn("Payment_Method",    F.trim(F.col("Payment_Method")))
    .withColumn("Device_Type",       F.trim(F.col("Device_Type")))
    .withColumn("Traffic_Source",    F.trim(F.col("Traffic_Source")))
    .withColumn("Membership_Status", F.trim(F.col("Membership_Status")))
    .withColumn("Shipping_Method",   F.trim(F.col("Shipping_Method")))
    .withColumn("Warehouse_Region",  F.trim(F.col("Warehouse_Region")))
    .withColumn("Order_Status",      F.trim(F.col("Order_Status")))

    # ერთიდაიგივე ინფორმაციის მომცემი სვეტების ამოღება
    .drop("Year", "Month", "Day", "Quarter", "Day_Of_Week", "Season")

    .select(
         "Order_ID"
        ,"Customer_ID"
        ,"Order_Date"
        ,"Country"
        ,"City"
        ,"Warehouse_Region"
        ,"Customer_Segment"
        ,"Membership_Status"
        ,"Customer_Age"
        ,"Customer_Gender"
        ,"Product_ID"
        ,"Product_Category"
        ,"Product_Subcategory"
        ,"Brand"
        ,"Unit_Price"
        ,"Quantity"
        ,"Discount_Percent"
        ,"Discount_Amount"
        ,"Coupon_Used"
        ,"Shipping_Cost"
        ,"Tax_Amount"
        ,"Order_Amount"
        ,"Profit_Margin_Percent"
        ,"Profit_Amount"
        ,"Payment_Method"
        ,"Device_Type"
        ,"Traffic_Source"
        ,"Shipping_Method"
        ,"Delivery_Days"
        ,"Order_Status"
        ,"Returned"
        ,"Review_Rating"
        ,"Customer_Lifetime_Value"
        ,"Holiday_Season"
        ,"High_Value_Order"
    )
)

display(orders_df.limit(5))

Orders rows: 30000


Order_ID,Customer_ID,Order_Date,Country,City,Warehouse_Region,Customer_Segment,Membership_Status,Customer_Age,Customer_Gender,Product_ID,Product_Category,Product_Subcategory,Brand,Unit_Price,Quantity,Discount_Percent,Discount_Amount,Coupon_Used,Shipping_Cost,Tax_Amount,Order_Amount,Profit_Margin_Percent,Profit_Amount,Payment_Method,Device_Type,Traffic_Source,Shipping_Method,Delivery_Days,Order_Status,Returned,Review_Rating,Customer_Lifetime_Value,Holiday_Season,High_Value_Order
615717,CUST007322,2023-01-01,Germany,Dubai,North,Loyal,Standard,32,Male,PROD02374,Books,Comics,PrimePlus,18.44,3,10,5.53,true,15.5,2.91,68.2,23.1,15.75,Debit Card,Mobile,Social Media,Express,2,Delivered,false,4.4,2144.92,false,false
626919,CUST004717,2023-01-01,France,London,East,Returning,Standard,50,Male,PROD01378,Sports,Equipment,PrimePlus,46.58,1,40,18.63,true,5.87,4.91,38.73,8.57,3.32,Wallet,Mobile,Email,Standard,9,Delivered,false,4.1,817.17,false,false
615781,CUST004415,2023-01-01,India,Berlin,Central,Returning,Standard,61,Male,PROD01850,Beauty,Skincare,Zenith,62.18,3,35,65.29,true,15.91,16.5,153.66,29.72,45.67,Cash on Delivery,Mobile,Paid Ads,Express,2,Delivered,false,5.0,541.16,false,false
621747,CUST004114,2023-01-01,United States,Riyadh,South,Returning,Standard,34,Male,PROD00545,Fashion,Women Clothing,FreshMart,91.06,3,0,0.0,false,9.17,16.02,298.37,25.22,75.25,Wallet,Desktop,Organic Search,Standard,6,Returned,true,3.4,700.49,false,false
625881,CUST000145,2023-01-01,India,Mumbai,Central,Premium,Standard,37,Male,PROD01398,Beauty,Makeup,Zenith,99.44,1,15,14.92,true,13.63,7.6,105.75,24.64,26.06,PayPal,Mobile,Referral,Express,8,Delivered,false,3.6,2133.77,false,false


In [0]:
# მომხმარებელთა განზომილება — თითო სტრიქონი თითოეულ უნიკალურ Customer_ID-ზე.

from pyspark.sql.window import Window

# ფანჯრული ფუნქციის გამოყენებით max Order_Date იღებს ბოლო ცნობილ მნიშვნელობებს, რადგან მომხმარებლის ატრიბუტები, როგორიცაა Membership_Status, დროთა განმავლობაში შეიძლება შეიცვალოს.
customer_window = Window.partitionBy("Customer_ID").orderBy(F.col("Order_Date").desc())

customers_df = (
    orders_df
    .withColumn("rn", F.row_number().over(customer_window))
    .filter(F.col("rn") == 1)
    .select(
         "Customer_ID"
        ,"Customer_Segment"
        ,"Membership_Status"
        ,"Customer_Age"
        ,"Customer_Gender"
        ,"Country"
        ,"City"
        ,"Customer_Lifetime_Value"
    )
)

display(customers_df.limit(5))

Unique customers: 8683


Customer_ID,Customer_Segment,Membership_Status,Customer_Age,Customer_Gender,Country,City,Customer_Lifetime_Value
CUST000001,Returning,Gold,29,Female,United States,Toronto,5720.0
CUST000002,Loyal,Silver,75,Female,India,Dubai,1046.02
CUST000003,New,Standard,34,Female,Pakistan,Berlin,3160.89
CUST000004,Loyal,Standard,30,Female,Germany,Mumbai,4619.94
CUST000005,Returning,Silver,25,Male,Canada,Paris,1707.8


In [0]:
# ვინახავთ ყველას getdata.calculated-ში
orders_df.write.mode("overwrite").saveAsTable("getdata.calculated.orders")
customers_df.write.mode("overwrite").saveAsTable("getdata.calculated.customers")

All three tables saved to getdata.calculated


In [0]:
# ვერიფიკაცია
for table in ["orders", "customers"]:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM getdata.calculated.{table}").first()["n"]
    print(f"getdata.calculated.{table}: {count} rows")

display(spark.sql("SELECT * FROM getdata.calculated.orders LIMIT 3"))
display(spark.sql("SELECT * FROM getdata.calculated.customers LIMIT 3"))

getdata.calculated.orders: 30000 rows
getdata.calculated.customers: 8683 rows


Order_ID,Customer_ID,Order_Date,Country,City,Warehouse_Region,Customer_Segment,Membership_Status,Customer_Age,Customer_Gender,Product_ID,Product_Category,Product_Subcategory,Brand,Unit_Price,Quantity,Discount_Percent,Discount_Amount,Coupon_Used,Shipping_Cost,Tax_Amount,Order_Amount,Profit_Margin_Percent,Profit_Amount,Payment_Method,Device_Type,Traffic_Source,Shipping_Method,Delivery_Days,Order_Status,Returned,Review_Rating,Customer_Lifetime_Value,Holiday_Season,High_Value_Order
615717,CUST007322,2023-01-01,Germany,Dubai,North,Loyal,Standard,32,Male,PROD02374,Books,Comics,PrimePlus,18.44,3,10,5.53,true,15.5,2.91,68.2,23.1,15.75,Debit Card,Mobile,Social Media,Express,2,Delivered,false,4.4,2144.92,false,false
626919,CUST004717,2023-01-01,France,London,East,Returning,Standard,50,Male,PROD01378,Sports,Equipment,PrimePlus,46.58,1,40,18.63,true,5.87,4.91,38.73,8.57,3.32,Wallet,Mobile,Email,Standard,9,Delivered,false,4.1,817.17,false,false
615781,CUST004415,2023-01-01,India,Berlin,Central,Returning,Standard,61,Male,PROD01850,Beauty,Skincare,Zenith,62.18,3,35,65.29,true,15.91,16.5,153.66,29.72,45.67,Cash on Delivery,Mobile,Paid Ads,Express,2,Delivered,false,5.0,541.16,false,false


Customer_ID,Customer_Segment,Membership_Status,Customer_Age,Customer_Gender,Country,City,Customer_Lifetime_Value
CUST000001,Returning,Gold,29,Female,United States,Toronto,5720.0
CUST000002,Loyal,Silver,75,Female,India,Dubai,1046.02
CUST000003,New,Standard,34,Female,Pakistan,Berlin,3160.89
